In [1]:
# DSBDA Practical 2: Data Wrangling II

import pandas as pd
import numpy as np

# Create Academic Performance Dataset
np.random.seed(50)

data = {
    'Student_id': range(1, 51),
    'Name': ['Student_' + str(i) for i in range(1, 51)],
    'Age': np.random.randint(18, 25, size=50),
    'Gender': np.random.choice(['Male', 'Female'], size=50),
    'Scores': [np.random.randint(50, 100, size=3).tolist() for _ in range(50)],
    'Attendance': np.random.randint(20, 100, size=50),
    'Grade': np.random.choice(['A', 'B', 'C', 'D', 'F'], size=50)
}

df = pd.DataFrame(data)

print("Initial Dataset:")
print(df.head(10))

# Function to assign grade based on average score
def assign_grade(scores):
    avg_score = np.mean(scores)
    if avg_score > 90:
        return 'A'
    elif avg_score > 80:
        return 'B'
    elif avg_score > 70:
        return 'C'
    elif avg_score > 60:
        return 'D'
    else:
        return 'F'

# Assign grades
df['Grade'] = df['Scores'].apply(assign_grade)

# Introduce missing values and inconsistencies
df.loc[8, 'Age'] = np.nan
df.loc[29, 'Age'] = np.nan
df.loc[35, 'Age'] = np.nan

df.loc[11, 'Scores'] = None
df.loc[19, 'Scores'] = None

df.loc[9, 'Attendance'] = 105      # invalid attendance
df.loc[15, 'Grade'] = 'Z'          # invalid grade

print("\nDataset with Missing Values and Inconsistencies:")
print(df.head(20))

# Scan for missing values and inconsistencies
missing_values = df.isnull().sum()
invalid_attendance = df[(df['Attendance'] < 0) | (df['Attendance'] > 100)]
invalid_grades = df[~df['Grade'].isin(['A', 'B', 'C', 'D', 'F'])]

print("\nMissing Values:\n", missing_values)
print("\nInvalid Attendance:\n", invalid_attendance)
print("\nInvalid Grades:\n", invalid_grades)

# Handle missing Age values
df['Age'] = df['Age'].fillna(df['Age'].median())

# Handle missing / invalid scores
def handle_invalid_scores(scores):
    if scores is None:
        return [0, 0, 0]
    return [max(0, min(100, s)) for s in scores]

df['Scores'] = df['Scores'].apply(handle_invalid_scores)

# Fix invalid attendance
df['Attendance'] = df['Attendance'].apply(
    lambda x: 100 if x > 100 else (0 if x < 0 else x)
)

# Reassign grades after cleaning
df['Grade'] = df['Scores'].apply(assign_grade)

print("\nDataset After Handling Missing Values and Inconsistencies:")
print(df.head(20))

# Introduce outliers
df.loc[5, 'Age'] = 65
df.loc[10, 'Attendance'] = 200
df.loc[12, 'Attendance'] = 166

print("\nDataset with Outliers:")
print(df.iloc[5:15])

# Show outliers before treatment
print("\nOutliers Before Treatment:")
print(df[(df['Age'] > 40) | (df['Attendance'] > 100)])

# Function to handle outliers using IQR
def handle_outliers_iqr(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    df[column] = df[column].apply(
        lambda x: upper_bound if x > upper_bound else
                  (lower_bound if x < lower_bound else x)
    )

# Handle outliers
handle_outliers_iqr(df, 'Age')
handle_outliers_iqr(df, 'Attendance')

print("\nDataset After Outlier Treatment:")
print(df.iloc[5:15])

# Data Transformation (Min-Max Scaling)
df['Scaled_Attendance'] = (
    (df['Attendance'] - df['Attendance'].min()) /
    (df['Attendance'].max() - df['Attendance'].min())
)

print("\nTransformation Applied: Attendance is normalized using Min-Max Scaling to bring values between 0 and 1 for better understanding and comparison.")

print("\nAttendance Before and After Scaling:")
print(df[['Attendance', 'Scaled_Attendance']].head(20))

print("\nFinal Cleaned Dataset:")
print(df.head(10))

Initial Dataset:
   Student_id        Name  Age  Gender        Scores  Attendance Grade
0           1   Student_1   18  Female  [64, 54, 72]          55     B
1           2   Student_2   18    Male  [93, 69, 82]          23     C
2           3   Student_3   21  Female  [87, 90, 80]          84     F
3           4   Student_4   23  Female  [94, 93, 85]          66     C
4           5   Student_5   19    Male  [88, 77, 78]          32     F
5           6   Student_6   24    Male  [81, 90, 65]          96     D
6           7   Student_7   22  Female  [55, 97, 54]          73     D
7           8   Student_8   24    Male  [54, 68, 97]          41     C
8           9   Student_9   23    Male  [92, 67, 76]          98     C
9          10  Student_10   24  Female  [58, 96, 61]          39     A

Dataset with Missing Values and Inconsistencies:
    Student_id        Name   Age  Gender        Scores  Attendance Grade
0            1   Student_1  18.0  Female  [64, 54, 72]          55     D
1     